# Инструмент просмотра спутниковых сцен Sentinel-3 SLSTR на траекториях полярных циклонов

**Выпускная квалификационная работа:**  
«Восстановление и анализ характеристик и траекторий полярных циклонов по данным дистанционного зондирования Земли»

**Автор:** Соловьева Яна Сергеевна  
**Факультет:** факультет географии и геоинформационных технологий, НИУ ВШЭ  
**Образовательная программа:** «География глобальных изменений и геоинформационные технологии»  
**Направление подготовки:** 05.03.02 «География»

## Назначение ноутбука

Этот ноутбук содержит приложение для визуальной проверки спутниковых сцен Sentinel-3 SLSTR, сопоставленных с траекториями полярных циклонов.

Главная задача приложения — показать, насколько корректно каждый спутниковый фрагмент связан с конкретным событием ПЦ, его временем съемки и положением на треке. Приложение используется как инструмент ручного контроля перед анализом траекторий и интерпретацией результатов модели восстановления характеристик.

В приложении можно просматривать:

1. спутниковый фрагмент ПЦ в масштабе карты;
2. положение сцены относительно полного трека циклона;
3. интерполированное положение центра ПЦ на момент съемки;
4. дату и время спутникового наблюдения;
5. спутник и идентификатор сцены;
6. координаты центра сцены и центра циклона;
7. значение центрального давления, связанное с данным наблюдением;
8. расстояние между положением сцены и треком;
9. последовательность наблюдений внутри одного события ПЦ.

In [3]:
# PL_viewer
# готовое приложение для просмотра сцен Sentinel-3 SLSTR на треках ПЦ

import subprocess
import sys
import os
import io
import math
import base64
import contextlib
import warnings

warnings.filterwarnings("ignore")


# 1. установка и импорты

def quiet_install(packages):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + packages,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=False,
    )


quiet_install(["ipywidgets", "folium", "pillow"])

import numpy as np
import pandas as pd
import folium
import ipywidgets as widgets

from PIL import Image
from IPython.display import display, clear_output

try:
    from google.colab import drive, output

    with contextlib.redirect_stdout(io.StringIO()):
        drive.mount("/content/drive", force_remount=False)

    output.enable_custom_widget_manager()

except Exception:
    pass


# 2. пути к данным

base_dir = "/content/drive/MyDrive/sentinel3/SLSTR"
project_dir = os.path.join(base_dir, "Solovieva_diploma_project")

scene_csv = os.path.join(project_dir, "Solovieva_diploma_PL_training_dataset_rematched_consistent.csv")
track_csv = os.path.join(project_dir, "Solovieva_diploma_PL_viewer_track_points_rematched_consistent.csv")

if not os.path.exists(scene_csv):
    raise FileNotFoundError("не найден файл со сценами: " + scene_csv)

if not os.path.exists(track_csv):
    raise FileNotFoundError("не найден файл с точками треков: " + track_csv)

# 3. настройки

km_per_pixel = 1.0
fallback_size_km = 512.0

scene_map_height = 760
all_tracks_map_height = 760

selected_scene_opacity = 0.92
other_scene_opacity = 0.28

# 4. базовые функции

def normalize_id(value):
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in ["nan", "none", "nat", "<na>"]:
        return ""

    if text.endswith(".0"):
        text = text[:-2]

    return text


def safe_float(value):
    try:
        value = float(value)
    except Exception:
        return np.nan

    if not np.isfinite(value):
        return np.nan

    return value


def fmt(value, digits=2):
    value = safe_float(value)

    if pd.isna(value):
        return "нет данных"

    return f"{value:.{digits}f}"


def lon_to_180(value):
    value = safe_float(value)

    if pd.isna(value):
        return np.nan

    return ((value + 180.0) % 360.0) - 180.0


def shortest_lon_delta(lon1, lon0):
    lon1 = lon_to_180(lon1)
    lon0 = lon_to_180(lon0)

    if pd.isna(lon1) or pd.isna(lon0):
        return np.nan

    return ((lon1 - lon0 + 180.0) % 360.0) - 180.0


def lon_near_center(lon, center_lon):
    center_lon = lon_to_180(center_lon)
    dlon = shortest_lon_delta(lon, center_lon)

    if pd.isna(center_lon) or pd.isna(dlon):
        return np.nan

    return center_lon + dlon


def circular_mean_lon(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().to_numpy()

    if len(values) == 0:
        return 0.0

    radians = np.deg2rad(values)
    s = np.nanmean(np.sin(radians))
    c = np.nanmean(np.cos(radians))

    if np.isclose(s, 0.0) and np.isclose(c, 0.0):
        return lon_to_180(np.nanmedian(values))

    return lon_to_180(np.rad2deg(np.arctan2(s, c)))


def parse_time_series(series):
    out = pd.to_datetime(series, errors="coerce", utc=True)
    return out.dt.tz_convert(None)


def parse_time(value):
    out = pd.to_datetime(value, errors="coerce", utc=True)

    if pd.isna(out):
        return pd.NaT

    try:
        return out.tz_convert(None)

    except Exception:
        out = pd.Timestamp(out)

        if out.tzinfo is not None:
            return out.tz_localize(None)

        return out


def fmt_time(value):
    value = parse_time(value)

    if pd.isna(value):
        return "нет данных"

    return value.strftime("%Y-%m-%d %H:%M:%S")


def resolve_path(value):
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text == "" or text.lower() in ["nan", "none", "nat", "<na>"]:
        return ""

    variants = [
        text,
        os.path.join(project_dir, text),
        os.path.join(base_dir, text),
    ]

    for path in variants:
        if os.path.exists(path):
            return path

    return ""


def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = safe_float(lat1)
    lon1 = lon_to_180(lon1)
    lat2 = safe_float(lat2)
    lon2 = lon_to_180(lon2)

    if any(pd.isna(x) for x in [lat1, lon1, lat2, lon2]):
        return np.nan

    r = 6371.0

    p1 = math.radians(lat1)
    p2 = math.radians(lat2)

    dp = math.radians(lat2 - lat1)

    dl = abs(lon2 - lon1)
    dl = min(dl, 360.0 - dl)
    dl = math.radians(dl)

    a = (
        math.sin(dp / 2.0) ** 2
        + math.cos(p1) * math.cos(p2) * math.sin(dl / 2.0) ** 2
    )

    return 2.0 * r * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))


def km_offsets_from_scene_center(scene_lat, scene_lon, point_lat, point_lon):
    scene_lat = safe_float(scene_lat)
    scene_lon = lon_to_180(scene_lon)
    point_lat = safe_float(point_lat)
    point_lon = lon_to_180(point_lon)

    if any(pd.isna(x) for x in [scene_lat, scene_lon, point_lat, point_lon]):
        return np.nan, np.nan

    dy_km = (point_lat - scene_lat) * 111.32
    dlon = shortest_lon_delta(point_lon, scene_lon)

    cos_lat = math.cos(math.radians(scene_lat))

    if abs(cos_lat) < 0.05:
        cos_lat = 0.05

    dx_km = dlon * 111.32 * cos_lat

    return dx_km, dy_km


def split_by_dateline(points):
    clean = []

    for lat, lon in points:
        lat = safe_float(lat)
        lon = lon_to_180(lon)

        if pd.notna(lat) and pd.notna(lon):
            clean.append([lat, lon])

    if len(clean) <= 1:
        return [clean] if clean else []

    segments = []
    segment = [clean[0]]

    for i in range(1, len(clean)):
        if abs(clean[i][1] - clean[i - 1][1]) > 180:
            segments.append(segment)
            segment = [clean[i]]
        else:
            segment.append(clean[i])

    segments.append(segment)

    return segments


def infer_satellite(row):
    for col in ["satellite", "platform", "platform_name", "spacecraft", "sensor"]:
        if col in row.index:
            text = str(row.get(col, "")).strip()

            if text and text.lower() not in ["nan", "none", "nat", "<na>"]:
                return text

    text = " ".join(
        str(row.get(col, ""))
        for col in [
            "product_id",
            "image_id",
            "source_png_filename",
            "source_png_path",
            "png_relative_path",
            "npy_relative_path",
            "scene_path",
        ]
        if col in row.index
    ).upper()

    if "S3A" in text or "SENTINEL-3A" in text:
        return "Sentinel-3A"

    if "S3B" in text or "SENTINEL-3B" in text:
        return "Sentinel-3B"

    if "SENTINEL-3" in text or "SLSTR" in text:
        return "Sentinel-3 SLSTR"

    return "нет данных"

# 5. чтение изображений

def scene_path_from_row(row):
    for col in [
        "png_path",
        "png_relative_path",
        "source_png_path",
        "scene_path_png",
        "scene_path_best",
        "scene_path",
    ]:
        if col in row.index:
            path = resolve_path(row[col])

            if path:
                return path

    for col in ["npy_path", "npy_relative_path", "scene_path_npy"]:
        if col in row.index:
            path = resolve_path(row[col])

            if path:
                return path

    return ""


def image_size(path):
    if not path or not os.path.exists(path):
        return np.nan, np.nan

    ext = os.path.splitext(path)[1].lower()

    if ext in [".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"]:
        with Image.open(path) as img:
            width, height = img.size

        return int(width), int(height)

    if ext == ".npy":
        arr = np.load(path, mmap_mode="r")
        shape = np.squeeze(arr).shape

        if len(shape) == 2:
            return int(shape[1]), int(shape[0])

        if len(shape) == 3:
            if shape[0] <= 4 and shape[1] > 16 and shape[2] > 16:
                return int(shape[2]), int(shape[1])

            return int(shape[1]), int(shape[0])

    return np.nan, np.nan


def scale_to_uint8(arr):
    arr = np.asarray(arr, dtype=np.float32)
    valid = np.isfinite(arr)

    if valid.sum() == 0:
        return np.zeros(arr.shape, dtype=np.uint8)

    fill_value = np.nanmedian(arr[valid])
    arr = np.where(np.isfinite(arr), arr, fill_value)

    p2 = np.nanpercentile(arr, 2)
    p98 = np.nanpercentile(arr, 98)

    if p98 <= p2:
        p2 = np.nanmin(arr)
        p98 = np.nanmax(arr)

    if p98 <= p2:
        return np.zeros(arr.shape, dtype=np.uint8)

    return ((arr - p2) / (p98 - p2) * 255).clip(0, 255).astype(np.uint8)


def read_scene_image(path):
    ext = os.path.splitext(path)[1].lower()

    if ext in [".png", ".jpg", ".jpeg", ".webp", ".tif", ".tiff"]:
        return Image.open(path).convert("RGB")

    if ext == ".npy":
        arr = np.squeeze(np.load(path).astype(np.float32))

        if arr.ndim == 2:
            return Image.fromarray(scale_to_uint8(arr), mode="L").convert("RGB")

        if arr.ndim == 3:
            if arr.shape[0] <= 4 and arr.shape[1] > 16 and arr.shape[2] > 16:
                arr = np.moveaxis(arr, 0, -1)

            if arr.shape[-1] == 1:
                return Image.fromarray(scale_to_uint8(arr[:, :, 0]), mode="L").convert("RGB")

            if arr.shape[-1] >= 3:
                rgb = np.stack(
                    [scale_to_uint8(arr[:, :, i]) for i in range(3)],
                    axis=-1,
                )

                return Image.fromarray(rgb, mode="RGB")

    raise ValueError("неподдерживаемый формат изображения: " + path)


def image_to_uri(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")

    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")

    return "data:image/png;base64," + encoded


def scene_uri(row):
    path = row.get("scene_path", "")

    if not path:
        return ""

    try:
        return image_to_uri(read_scene_image(path))

    except Exception:
        return ""


def small_scene_image(row, max_size=470):
    path = row.get("scene_path", "")

    if not path:
        return None

    img = read_scene_image(path).copy()
    img.thumbnail((max_size, max_size), Image.LANCZOS)

    return img

# 6. загрузка и подготовка таблиц

scenes = pd.read_csv(scene_csv, dtype=str)
tracks = pd.read_csv(track_csv, dtype=str)

required_scene_cols = [
    "ID",
    "step",
    "datetime",
    "lat",
    "lon",
    "slp",
    "rel_vort_850_smth",
    "scene_datetime",
    "scene_lat",
    "scene_lon",
]

required_track_cols = [
    "ID",
    "step",
    "track_datetime",
    "lat",
    "lon",
    "slp",
    "rel_vort_850_smth",
]

missing_scene = [col for col in required_scene_cols if col not in scenes.columns]
missing_track = [col for col in required_track_cols if col not in tracks.columns]

if missing_scene:
    raise ValueError("в таблице сцен не хватает колонок: " + ", ".join(missing_scene))

if missing_track:
    raise ValueError("в таблице треков не хватает колонок: " + ", ".join(missing_track))

scenes = scenes.copy()
tracks = tracks.copy()

scenes["ID"] = scenes["ID"].map(normalize_id)
tracks["ID"] = tracks["ID"].map(normalize_id)

scenes["step"] = pd.to_numeric(scenes["step"], errors="coerce")
scenes["datetime"] = parse_time_series(scenes["datetime"])
scenes["lat"] = pd.to_numeric(scenes["lat"], errors="coerce")
scenes["lon"] = pd.to_numeric(scenes["lon"], errors="coerce").map(lon_to_180)
scenes["slp"] = pd.to_numeric(scenes["slp"], errors="coerce")
scenes["rel_vort_850_smth"] = pd.to_numeric(scenes["rel_vort_850_smth"], errors="coerce")

scenes["scene_datetime"] = parse_time_series(scenes["scene_datetime"])
scenes["scene_lat"] = pd.to_numeric(scenes["scene_lat"], errors="coerce")
scenes["scene_lon"] = pd.to_numeric(scenes["scene_lon"], errors="coerce").map(lon_to_180)

if "original_ID" not in scenes.columns:
    scenes["original_ID"] = scenes["ID"]

if "image_id" not in scenes.columns:
    scenes["image_id"] = scenes.index.astype(str)

if "source_png_filename" not in scenes.columns:
    scenes["source_png_filename"] = ""

scenes["plot_datetime"] = scenes["scene_datetime"]
scenes["plot_lat"] = scenes["scene_lat"]
scenes["plot_lon"] = scenes["scene_lon"]

scenes["assigned_track_datetime"] = scenes["datetime"]
scenes["assigned_track_lat"] = scenes["lat"]
scenes["assigned_track_lon"] = scenes["lon"]
scenes["assigned_track_slp"] = scenes["slp"]
scenes["assigned_track_rel_vort_850_smth"] = scenes["rel_vort_850_smth"]

scenes["satellite_clean"] = scenes.apply(infer_satellite, axis=1)
scenes["scene_path"] = scenes.apply(scene_path_from_row, axis=1)
scenes["scene_exists"] = scenes["scene_path"].map(lambda x: os.path.exists(x) if x else False)

sizes = scenes["scene_path"].map(image_size)

scenes["image_width_px"] = [x[0] for x in sizes]
scenes["image_height_px"] = [x[1] for x in sizes]

scenes["image_width_px"] = pd.to_numeric(scenes["image_width_px"], errors="coerce")
scenes["image_height_px"] = pd.to_numeric(scenes["image_height_px"], errors="coerce")

scenes["scene_width_km"] = scenes["image_width_px"] * km_per_pixel
scenes["scene_height_km"] = scenes["image_height_px"] * km_per_pixel

scenes.loc[scenes["scene_width_km"].isna(), "scene_width_km"] = fallback_size_km
scenes.loc[scenes["scene_height_km"].isna(), "scene_height_km"] = fallback_size_km

scenes = scenes[
    scenes["scene_exists"]
    & scenes["ID"].notna()
    & (scenes["ID"] != "")
    & scenes["step"].notna()
    & scenes["plot_datetime"].notna()
    & scenes["plot_lat"].notna()
    & scenes["plot_lon"].notna()
].copy()

if len(scenes) == 0:
    raise ValueError("после фильтрации не осталось сцен с найденными файлами изображений")

scenes = scenes.sort_values(["ID", "plot_datetime", "step", "image_id"]).reset_index(drop=True)
scenes["viewer_index"] = np.arange(len(scenes))

tracks["step"] = pd.to_numeric(tracks["step"], errors="coerce")
tracks["track_datetime"] = parse_time_series(tracks["track_datetime"])
tracks["lat"] = pd.to_numeric(tracks["lat"], errors="coerce")
tracks["lon"] = pd.to_numeric(tracks["lon"], errors="coerce").map(lon_to_180)
tracks["slp"] = pd.to_numeric(tracks["slp"], errors="coerce")
tracks["rel_vort_850_smth"] = pd.to_numeric(tracks["rel_vort_850_smth"], errors="coerce")

tracks = tracks[
    tracks["ID"].notna()
    & (tracks["ID"] != "")
    & tracks["step"].notna()
    & tracks["track_datetime"].notna()
    & tracks["lat"].notna()
    & tracks["lon"].notna()
].copy()

tracks = tracks.sort_values(["ID", "track_datetime", "step"]).reset_index(drop=True)

track_groups = {}

for cyclone_id, group in tracks.groupby("ID"):
    group = group.copy()
    group = group.sort_values(["track_datetime", "step"])
    group = group.drop_duplicates("track_datetime", keep="first")
    group = group.reset_index(drop=True)

    track_groups[str(cyclone_id)] = group

# 7. интерполяция положения сцены на полном треке

def interpolate_lon(lon0, lon1, frac):
    lon0 = lon_to_180(lon0)
    lon1 = lon_to_180(lon1)
    frac = safe_float(frac)

    if pd.isna(lon0) or pd.isna(lon1) or pd.isna(frac):
        return np.nan

    dlon = shortest_lon_delta(lon1, lon0)

    if pd.isna(dlon):
        return np.nan

    return lon_to_180(lon0 + frac * dlon)


def nearest_track_position(group, scene_time, reason):
    if len(group) == 0 or pd.isna(scene_time):
        return {
            "track_time_lat": np.nan,
            "track_time_lon": np.nan,
            "track_time_prev_step": np.nan,
            "track_time_next_step": np.nan,
            "track_time_prev_datetime": pd.NaT,
            "track_time_next_datetime": pd.NaT,
            "track_time_fraction": np.nan,
            "track_time_method": reason,
        }

    scene_ns = int(pd.Timestamp(scene_time).value)
    time_ns = group["track_datetime"].map(lambda x: int(pd.Timestamp(x).value)).to_numpy()

    nearest_i = int(np.argmin(np.abs(time_ns - scene_ns)))
    p = group.iloc[nearest_i]

    return {
        "track_time_lat": safe_float(p["lat"]),
        "track_time_lon": lon_to_180(p["lon"]),
        "track_time_prev_step": safe_float(p["step"]),
        "track_time_next_step": safe_float(p["step"]),
        "track_time_prev_datetime": p["track_datetime"],
        "track_time_next_datetime": p["track_datetime"],
        "track_time_fraction": 0.0,
        "track_time_method": reason,
    }


def interpolate_scene_on_track(cyclone_id, scene_time):
    cyclone_id = normalize_id(cyclone_id)
    scene_time = parse_time(scene_time)

    empty = {
        "track_time_lat": np.nan,
        "track_time_lon": np.nan,
        "track_time_prev_step": np.nan,
        "track_time_next_step": np.nan,
        "track_time_prev_datetime": pd.NaT,
        "track_time_next_datetime": pd.NaT,
        "track_time_fraction": np.nan,
        "track_time_method": "нет трека или времени сцены",
    }

    if cyclone_id == "" or pd.isna(scene_time):
        return empty

    if cyclone_id not in track_groups:
        return empty

    group = track_groups[cyclone_id].copy()

    if len(group) == 0:
        return empty

    if len(group) == 1:
        return nearest_track_position(group, scene_time, "одна точка трека")

    time_ns = group["track_datetime"].map(lambda x: int(pd.Timestamp(x).value)).to_numpy()
    scene_ns = int(pd.Timestamp(scene_time).value)

    if scene_ns <= time_ns[0]:
        return nearest_track_position(group.iloc[[0]].copy(), scene_time, "первая точка трека")

    if scene_ns >= time_ns[-1]:
        return nearest_track_position(group.iloc[[-1]].copy(), scene_time, "последняя точка трека")

    next_i = int(np.searchsorted(time_ns, scene_ns, side="right"))
    prev_i = next_i - 1

    p0 = group.iloc[prev_i]
    p1 = group.iloc[next_i]

    t0 = int(time_ns[prev_i])
    t1 = int(time_ns[next_i])

    if t1 == t0:
        return nearest_track_position(group.iloc[[prev_i]].copy(), scene_time, "совпадающие времена точек трека")

    frac = (scene_ns - t0) / (t1 - t0)

    lat0 = safe_float(p0["lat"])
    lat1 = safe_float(p1["lat"])
    lon0 = lon_to_180(p0["lon"])
    lon1 = lon_to_180(p1["lon"])

    lat = lat0 + frac * (lat1 - lat0)
    lon = interpolate_lon(lon0, lon1, frac)

    if pd.isna(lat) or pd.isna(lon):
        return nearest_track_position(group, scene_time, "ближайшая точка трека, интерполяция дала пустые координаты")

    return {
        "track_time_lat": lat,
        "track_time_lon": lon,
        "track_time_prev_step": safe_float(p0["step"]),
        "track_time_next_step": safe_float(p1["step"]),
        "track_time_prev_datetime": p0["track_datetime"],
        "track_time_next_datetime": p1["track_datetime"],
        "track_time_fraction": frac,
        "track_time_method": "линейная интерполяция по времени",
    }


interpolated = scenes.apply(
    lambda row: pd.Series(
        interpolate_scene_on_track(
            row["ID"],
            row["plot_datetime"],
        )
    ),
    axis=1,
)

for col in interpolated.columns:
    if col in scenes.columns:
        scenes = scenes.drop(columns=[col])

scenes = pd.concat([scenes, interpolated], axis=1)

scenes["distance_scene_center_to_track_time_km"] = scenes.apply(
    lambda row: haversine_km(
        row["plot_lat"],
        row["plot_lon"],
        row["track_time_lat"],
        row["track_time_lon"],
    ),
    axis=1,
)

scenes["offset_px"] = scenes["distance_scene_center_to_track_time_km"] / km_per_pixel
scenes["share_width"] = scenes["distance_scene_center_to_track_time_km"] / scenes["scene_width_km"] * 100
scenes["half_diagonal_km"] = np.sqrt(scenes["scene_width_km"] ** 2 + scenes["scene_height_km"] ** 2) / 2.0


def scene_position_status(row):
    dx_km, dy_km = km_offsets_from_scene_center(
        row["plot_lat"],
        row["plot_lon"],
        row["track_time_lat"],
        row["track_time_lon"],
    )

    if pd.isna(dx_km) or pd.isna(dy_km):
        return "нет данных"

    half_w = safe_float(row["scene_width_km"]) / 2.0
    half_h = safe_float(row["scene_height_km"]) / 2.0

    if pd.isna(half_w) or pd.isna(half_h):
        return "нет данных"

    if abs(dx_km) <= half_w and abs(dy_km) <= half_h:
        return "точка трека попадает в пределы сцены"

    return "точка трека вне сцены"


scenes["position_status"] = scenes.apply(scene_position_status, axis=1)

# 8. геометрия, всплывающие окна и таблицы

def scene_bounds(row, display_center_lon):
    lat = safe_float(row.get("plot_lat"))
    lon = lon_near_center(row.get("plot_lon"), display_center_lon)
    width_km = safe_float(row.get("scene_width_km"))
    height_km = safe_float(row.get("scene_height_km"))

    if any(pd.isna(x) for x in [lat, lon, width_km, height_km]):
        return None

    lat_half = (height_km / 2.0) / 111.32

    cos_lat = math.cos(math.radians(lat))

    if abs(cos_lat) < 0.05:
        cos_lat = 0.05

    lon_half = (width_km / 2.0) / (111.32 * cos_lat)

    return [
        [lat - lat_half, lon - lon_half],
        [lat + lat_half, lon + lon_half],
    ]


def fit_map(m, points):
    clean = []

    for item in points:
        if isinstance(item, (list, tuple)) and len(item) == 2:
            if isinstance(item[0], (list, tuple)):
                for sub in item:
                    if isinstance(sub, (list, tuple)) and len(sub) == 2:
                        lat = safe_float(sub[0])
                        lon = safe_float(sub[1])

                        if pd.notna(lat) and pd.notna(lon):
                            clean.append([lat, lon])
            else:
                lat = safe_float(item[0])
                lon = safe_float(item[1])

                if pd.notna(lat) and pd.notna(lon):
                    clean.append([lat, lon])

    if len(clean) == 0:
        return

    lats = [p[0] for p in clean]
    lons = [p[1] for p in clean]

    if len(clean) == 1:
        lat = lats[0]
        lon = lons[0]
        m.fit_bounds([[lat - 1, lon - 1], [lat + 1, lon + 1]])
    else:
        m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])


def scene_popup(row):
    return f"""
    <div style="font-family:Arial; font-size:12px; width:360px; line-height:1.45;">
      <b>сцена SLSTR</b><br>
      ID: {row.get('ID', '')}<br>
      step: {fmt(row.get('step'), 0)}<br>
      image_id: {row.get('image_id', '')}<br>
      время съемки: {fmt_time(row.get('plot_datetime'))}<br>
      спутник: {row.get('satellite_clean', 'нет данных')}<br>
      центр сцены: {fmt(row.get('plot_lat'), 5)}, {fmt(row.get('plot_lon'), 5)}<br>
      давление: {fmt(row.get('assigned_track_slp'), 2)} гПа<br>
      завихренность 850 гПа: {fmt(row.get('assigned_track_rel_vort_850_smth'), 3)}<br>
      размер: {fmt(row.get('scene_width_km'), 0)} x {fmt(row.get('scene_height_km'), 0)} км
    </div>
    """


def track_popup(point):
    return f"""
    <div style="font-family:Arial; font-size:12px; width:320px; line-height:1.45;">
      <b>точка полного трека</b><br>
      ID: {point.get('ID', '')}<br>
      step: {fmt(point.get('step'), 0)}<br>
      время: {fmt_time(point.get('track_datetime'))}<br>
      координаты: {fmt(point.get('lat'), 5)}, {fmt(point.get('lon'), 5)}<br>
      давление: {fmt(point.get('slp'), 2)} гПа<br>
      завихренность 850 гПа: {fmt(point.get('rel_vort_850_smth'), 3)}
    </div>
    """


def interpolated_popup(row):
    return f"""
    <div style="font-family:Arial; font-size:12px; width:370px; line-height:1.45;">
      <b>положение сцены на полном треке по времени съемки</b><br>
      ID: {row.get('ID', '')}<br>
      время сцены: {fmt_time(row.get('plot_datetime'))}<br>
      координаты на треке: {fmt(row.get('track_time_lat'), 5)}, {fmt(row.get('track_time_lon'), 5)}<br>
      между step {fmt(row.get('track_time_prev_step'), 0)} и {fmt(row.get('track_time_next_step'), 0)}<br>
      доля между точками: {fmt(row.get('track_time_fraction'), 3)}<br>
      метод: {row.get('track_time_method', '')}<br>
      расстояние от центра сцены: {fmt(row.get('distance_scene_center_to_track_time_km'), 1)} км
    </div>
    """


def scene_info_widget(row):
    html = f"""
    <div style="font-family:Arial; font-size:13px; line-height:1.48; border:1px solid #ddd; border-radius:12px; padding:12px; background:#fff;">
      <div style="font-size:17px; font-weight:700; margin-bottom:8px;">сцена Sentinel-3 SLSTR</div>

      <b>ID:</b> {row.get('ID', '')}<br>
      <b>step:</b> {fmt(row.get('step'), 0)}<br>
      <b>image_id:</b> {row.get('image_id', '')}<br>
      <b>спутник:</b> {row.get('satellite_clean', 'нет данных')}<br>
      <b>время съемки:</b> {fmt_time(row.get('plot_datetime'))}<br>
      <b>центр сцены:</b> {fmt(row.get('plot_lat'), 5)}, {fmt(row.get('plot_lon'), 5)}<br>

      <br>
      <b>характеристики, присвоенные сцене:</b><br>
      давление: {fmt(row.get('assigned_track_slp'), 2)} гПа<br>
      завихренность 850 гПа: {fmt(row.get('assigned_track_rel_vort_850_smth'), 3)}<br>
      время точки характеристик: {fmt_time(row.get('assigned_track_datetime'))}<br>
      координаты точки характеристик: {fmt(row.get('assigned_track_lat'), 5)}, {fmt(row.get('assigned_track_lon'), 5)}<br>

      <br>
      <b>положение на полном треке по времени съемки:</b><br>
      координаты: {fmt(row.get('track_time_lat'), 5)}, {fmt(row.get('track_time_lon'), 5)}<br>
      между step {fmt(row.get('track_time_prev_step'), 0)} и {fmt(row.get('track_time_next_step'), 0)}<br>
      доля между точками: {fmt(row.get('track_time_fraction'), 3)}<br>
      метод: {row.get('track_time_method', '')}<br>

      <br>
      <b>контроль совпадения:</b><br>
      расстояние от центра сцены до трека: {fmt(row.get('distance_scene_center_to_track_time_km'), 1)} км<br>
      смещение: {fmt(row.get('offset_px'), 1)} px при 1 км/px<br>
      доля от ширины сцены: {fmt(row.get('share_width'), 1)}%<br>
      положение: {row.get('position_status', '')}<br>

      <br>
      <b>размер сцены:</b> {fmt(row.get('image_width_px'), 0)} x {fmt(row.get('image_height_px'), 0)} px; {fmt(row.get('scene_width_km'), 0)} x {fmt(row.get('scene_height_km'), 0)} км<br>
      <b>файл:</b> {row.get('source_png_filename', '')}
    </div>
    """

    return widgets.HTML(value=html)


def scene_label(row):
    return (
        f"{int(row['viewer_index']) + 1}/{len(scenes)} | "
        f"ID {row['ID']} | "
        f"step {fmt(row['step'], 0)} | "
        f"{fmt_time(row['plot_datetime'])} | "
        f"slp={fmt(row['assigned_track_slp'], 1)} гПа | "
        f"dist={fmt(row['distance_scene_center_to_track_time_km'], 1)} км"
    )


def id_label(cyclone_id):
    subset = scenes[scenes["ID"].astype(str) == str(cyclone_id)]

    if len(subset) == 0:
        return f"ID {cyclone_id}"

    return (
        f"ID {cyclone_id} | сцен={len(subset)} | "
        f"slp={subset['assigned_track_slp'].min():.1f}–{subset['assigned_track_slp'].max():.1f} гПа"
    )


def compact_scene_table(cyclone_id):
    subset = scenes[scenes["ID"].astype(str) == str(cyclone_id)].copy()

    cols = [
        "viewer_index",
        "ID",
        "step",
        "plot_datetime",
        "satellite_clean",
        "plot_lat",
        "plot_lon",
        "assigned_track_slp",
        "assigned_track_rel_vort_850_smth",
        "track_time_lat",
        "track_time_lon",
        "distance_scene_center_to_track_time_km",
        "offset_px",
        "share_width",
        "position_status",
    ]

    cols = [col for col in cols if col in subset.columns]
    out = subset[cols].copy()

    out = out.rename(
        columns={
            "viewer_index": "индекс",
            "plot_datetime": "время съемки",
            "satellite_clean": "спутник",
            "plot_lat": "lat сцены",
            "plot_lon": "lon сцены",
            "assigned_track_slp": "давление, гПа",
            "assigned_track_rel_vort_850_smth": "завихренность 850 гПа",
            "track_time_lat": "lat на треке по времени",
            "track_time_lon": "lon на треке по времени",
            "distance_scene_center_to_track_time_km": "расстояние до трека, км",
            "offset_px": "смещение, px",
            "share_width": "доля от ширины сцены, %",
            "position_status": "проверка",
        }
    )

    return out.reset_index(drop=True)

# 9. карты

def add_track_to_scene_map(m, cyclone_id, display_center_lon, show_points):
    one_track = tracks[tracks["ID"].astype(str) == str(cyclone_id)].copy()
    one_track = one_track.sort_values(["track_datetime", "step"]).reset_index(drop=True)

    if len(one_track) == 0:
        return []

    one_track["display_lon"] = one_track["lon"].map(lambda x: lon_near_center(x, display_center_lon))
    points = one_track[["lat", "display_lon"]].dropna().values.tolist()

    if len(points) >= 2:
        folium.PolyLine(
            points,
            color="#111111",
            weight=3,
            opacity=0.9,
            tooltip=f"полный трек ID {cyclone_id}",
        ).add_to(m)

    if show_points:
        for _, point in one_track.iterrows():
            folium.CircleMarker(
                location=[point["lat"], point["display_lon"]],
                radius=4,
                color="#111111",
                fill=True,
                fill_color="#111111",
                fill_opacity=0.75,
                weight=1,
                tooltip=f"step {fmt(point.get('step'), 0)} | {fmt_time(point.get('track_datetime'))}",
                popup=folium.Popup(track_popup(point), max_width=360),
            ).add_to(m)

    return points


def add_scene_overlay(m, row, display_center_lon, selected=True, show_image=True):
    bounds = scene_bounds(row, display_center_lon)

    if bounds is None:
        return []

    color = "#d62728" if selected else "#9467bd"
    weight = 4 if selected else 2
    opacity = selected_scene_opacity if selected else other_scene_opacity

    if show_image:
        uri = scene_uri(row)

        if uri:
            folium.raster_layers.ImageOverlay(
                image=uri,
                bounds=bounds,
                opacity=opacity,
                interactive=True,
                cross_origin=False,
                zindex=5 if selected else 2,
            ).add_to(m)

    folium.Rectangle(
        bounds=bounds,
        color=color,
        weight=weight,
        fill=False,
        tooltip="выбранная сцена" if selected else "другая сцена этого трека",
        popup=folium.Popup(scene_popup(row), max_width=410),
    ).add_to(m)

    return bounds


def add_connection(m, row, display_center_lon):
    scene_lat = safe_float(row.get("plot_lat"))
    scene_lon = lon_near_center(row.get("plot_lon"), display_center_lon)

    track_lat = safe_float(row.get("track_time_lat"))
    track_lon = lon_near_center(row.get("track_time_lon"), display_center_lon)

    if any(pd.isna(x) for x in [scene_lat, scene_lon, track_lat, track_lon]):
        return []

    folium.PolyLine(
        [[scene_lat, scene_lon], [track_lat, track_lon]],
        color="#1f77b4",
        weight=3,
        opacity=0.9,
        dash_array="7,7",
        tooltip="смещение центра сцены относительно положения на треке по времени съемки",
    ).add_to(m)

    folium.CircleMarker(
        [scene_lat, scene_lon],
        radius=7,
        color="#d62728",
        fill=True,
        fill_color="#d62728",
        fill_opacity=0.95,
        weight=2,
        tooltip="центр сцены",
        popup=folium.Popup(scene_popup(row), max_width=410),
    ).add_to(m)

    folium.CircleMarker(
        [track_lat, track_lon],
        radius=7,
        color="#1f77b4",
        fill=True,
        fill_color="#1f77b4",
        fill_opacity=0.95,
        weight=2,
        tooltip="положение сцены на треке по времени съемки",
        popup=folium.Popup(interpolated_popup(row), max_width=410),
    ).add_to(m)

    return [[scene_lat, scene_lon], [track_lat, track_lon]]


def build_scene_map(row, show_other_frames=False, show_other_scenes=False, show_track_points=False):
    display_center_lon = circular_mean_lon(
        [
            row.get("plot_lon"),
            row.get("track_time_lon"),
            row.get("assigned_track_lon"),
        ]
    )

    center_lat = safe_float(row.get("plot_lat"))
    center_lon = lon_near_center(row.get("plot_lon"), display_center_lon)

    if pd.isna(center_lat) or pd.isna(center_lon):
        center_lat = 70.0
        center_lon = 0.0

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=5,
        tiles="OpenStreetMap",
        control_scale=True,
        zoom_control=True,
        scroll_wheel_zoom=False,
        double_click_zoom=False,
        box_zoom=False,
        keyboard=False,
        width="100%",
        height=f"{scene_map_height}px",
        world_copy_jump=False,
    )

    fit_items = []

    fit_items += add_track_to_scene_map(
        m,
        row.get("ID"),
        display_center_lon,
        show_track_points,
    )

    if show_other_frames or show_other_scenes:
        other = scenes[
            (scenes["ID"].astype(str) == str(row.get("ID")))
            & (scenes["viewer_index"] != row.get("viewer_index"))
        ]

        for _, other_row in other.iterrows():
            fit_items += add_scene_overlay(
                m,
                other_row,
                display_center_lon,
                selected=False,
                show_image=show_other_scenes,
            )

    fit_items += add_scene_overlay(
        m,
        row,
        display_center_lon,
        selected=True,
        show_image=True,
    )

    fit_items += add_connection(m, row, display_center_lon)

    fit_map(m, fit_items)

    return m

# 10. сводка по трекам

scene_counts = scenes.groupby("ID").size().rename("n_scenes")

track_summary = (
    tracks.groupby("ID")
    .agg(
        n_track_points=("step", "count"),
        first_time=("track_datetime", "min"),
        last_time=("track_datetime", "max"),
        slp_min=("slp", "min"),
        slp_max=("slp", "max"),
        slp_mean=("slp", "mean"),
        vort_min=("rel_vort_850_smth", "min"),
        vort_max=("rel_vort_850_smth", "max"),
        vort_mean=("rel_vort_850_smth", "mean"),
    )
    .reset_index()
    .merge(scene_counts.reset_index(), on="ID", how="inner")
    .sort_values(["n_scenes", "n_track_points"], ascending=[False, False])
    .reset_index(drop=True)
)

all_tracks_points = tracks[tracks["ID"].astype(str).isin(track_summary["ID"].astype(str))].copy()


def all_tracks_table():
    out = track_summary.copy()

    return out.rename(
        columns={
            "n_scenes": "сцен",
            "n_track_points": "точек полного трека",
            "first_time": "начало",
            "last_time": "конец",
            "slp_min": "мин. давление",
            "slp_max": "макс. давление",
            "slp_mean": "ср. давление",
            "vort_min": "мин. завихренность",
            "vort_max": "макс. завихренность",
            "vort_mean": "ср. завихренность",
        }
    )


def track_summary_popup(track_id):
    row = track_summary[track_summary["ID"].astype(str) == str(track_id)].iloc[0]

    return f"""
    <div style="font-family:Arial; font-size:12px; width:340px; line-height:1.45;">
      <b>полный трек ПЦ</b><br>
      ID: {row.get('ID', '')}<br>
      сцен: {int(row.get('n_scenes', 0))}<br>
      точек полного трека: {int(row.get('n_track_points', 0))}<br>
      период: {fmt_time(row.get('first_time'))} — {fmt_time(row.get('last_time'))}<br>
      давление: {fmt(row.get('slp_min'), 1)} — {fmt(row.get('slp_max'), 1)} гПа
    </div>
    """


def build_all_tracks_map(active_id="__all__", show_points=True):
    if active_id == "__all__":
        points_df = all_tracks_points.copy()
    else:
        points_df = all_tracks_points[all_tracks_points["ID"].astype(str) == str(active_id)].copy()

        if len(points_df) == 0:
            points_df = all_tracks_points.copy()

    center_lat = float(points_df["lat"].median())
    center_lon = circular_mean_lon(points_df["lon"])

    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=3,
        tiles="OpenStreetMap",
        control_scale=True,
        zoom_control=True,
        scroll_wheel_zoom=True,
        width="100%",
        height=f"{all_tracks_map_height}px",
        world_copy_jump=False,
    )

    if active_id == "__all__":
        ids_to_draw = track_summary["ID"].astype(str).tolist()

        for track_id in ids_to_draw:
            one_track = all_tracks_points[all_tracks_points["ID"].astype(str) == str(track_id)].copy()
            one_track = one_track.sort_values(["track_datetime", "step"])

            points = one_track[["lat", "lon"]].dropna().values.tolist()

            for segment in split_by_dateline(points):
                if len(segment) >= 2:
                    folium.PolyLine(
                        segment,
                        color="#333333",
                        weight=2,
                        opacity=0.42,
                        tooltip=f"полный трек ID {track_id}",
                        popup=folium.Popup(track_summary_popup(track_id), max_width=370),
                    ).add_to(m)

            if show_points:
                for _, p in one_track.iterrows():
                    folium.CircleMarker(
                        [p["lat"], p["lon"]],
                        radius=3,
                        color="#333333",
                        fill=True,
                        fill_color="#333333",
                        fill_opacity=0.55,
                        weight=1,
                        tooltip=f"ID {p.get('ID')} | {fmt_time(p.get('track_datetime'))} | slp={fmt(p.get('slp'), 1)} гПа",
                        popup=folium.Popup(track_popup(p), max_width=340),
                    ).add_to(m)

    else:
        track_id = str(active_id)
        one_track = all_tracks_points[all_tracks_points["ID"].astype(str) == track_id].copy()
        one_track = one_track.sort_values(["track_datetime", "step"])

        display_center_lon = circular_mean_lon(one_track["lon"])
        one_track["display_lon"] = one_track["lon"].map(lambda x: lon_near_center(x, display_center_lon))

        points = one_track[["lat", "display_lon"]].dropna().values.tolist()

        if len(points) >= 2:
            folium.PolyLine(
                points,
                color="#d62728",
                weight=4,
                opacity=0.95,
                tooltip=f"полный трек ID {track_id}",
                popup=folium.Popup(track_summary_popup(track_id), max_width=370),
            ).add_to(m)

        if show_points:
            for _, p in one_track.iterrows():
                folium.CircleMarker(
                    [p["lat"], p["display_lon"]],
                    radius=5,
                    color="#d62728",
                    fill=True,
                    fill_color="#d62728",
                    fill_opacity=0.9,
                    weight=1,
                    tooltip=f"ID {p.get('ID')} | {fmt_time(p.get('track_datetime'))} | slp={fmt(p.get('slp'), 1)} гПа",
                    popup=folium.Popup(track_popup(p), max_width=340),
                ).add_to(m)

        fit_map(m, points)

    return m

# 11. виджеты вкладки проверки сцен

id_values = sorted(
    scenes["ID"].astype(str).unique().tolist(),
    key=lambda x: (-int((scenes["ID"].astype(str) == x).sum()), x),
)

id_dropdown = widgets.Dropdown(
    options=[(id_label(x), x) for x in id_values],
    value=id_values[0],
    description="трек",
    layout=widgets.Layout(width="850px"),
)

scene_dropdown = widgets.Dropdown(
    options=[],
    description="сцена",
    layout=widgets.Layout(width="1220px"),
)

prev_button = widgets.Button(description="назад по треку", layout=widgets.Layout(width="140px"))
next_button = widgets.Button(description="вперед по треку", layout=widgets.Layout(width="150px"))
global_prev_button = widgets.Button(description="предыдущая", layout=widgets.Layout(width="125px"))
global_next_button = widgets.Button(description="следующая", layout=widgets.Layout(width="125px"))

show_other_frames_checkbox = widgets.Checkbox(
    value=False,
    description="рамки других сцен этого трека",
    layout=widgets.Layout(width="260px"),
)

show_other_scenes_checkbox = widgets.Checkbox(
    value=False,
    description="другие сцены трека",
    layout=widgets.Layout(width="190px"),
)

show_track_points_checkbox = widgets.Checkbox(
    value=False,
    description="точки полного трека",
    layout=widgets.Layout(width="180px"),
)

status_out = widgets.Output()
info_out = widgets.Output()
image_out = widgets.Output()
map_out = widgets.Output()
table_out = widgets.Output()


def indices_for_id(cyclone_id):
    return scenes.index[scenes["ID"].astype(str) == str(cyclone_id)].tolist()


def global_indices():
    return scenes.sort_values(["plot_datetime", "ID", "step", "image_id"]).index.tolist()


def update_scene_dropdown(preferred_index=None):
    indices = indices_for_id(id_dropdown.value)

    scene_dropdown.options = [(scene_label(scenes.loc[i]), int(i)) for i in indices]

    if len(indices) == 0:
        scene_dropdown.value = None
    elif preferred_index in indices:
        scene_dropdown.value = int(preferred_index)
    else:
        scene_dropdown.value = int(indices[0])


def display_selected_scene(change=None):
    if scene_dropdown.value is None:
        return

    row = scenes.loc[int(scene_dropdown.value)]

    with status_out:
        clear_output(wait=True)

        display(
            widgets.HTML(
                value=f"""
                <div style="font-family:Arial; font-size:13px; line-height:1.45; padding:8px 10px; border:1px solid #ddd; border-radius:10px; background:#f7f7f7;">
                  <b>выбрана сцена:</b> {scene_label(row)}<br>
                  <b>проверка:</b> {row.get('position_status', '')}
                </div>
                """
            )
        )

    with info_out:
        clear_output(wait=True)
        display(scene_info_widget(row))

    with image_out:
        clear_output(wait=True)

        try:
            img = small_scene_image(row, max_size=470)
            display(widgets.HTML(value="<b>миниатюра сцены</b>"))

            if img is not None:
                display(img)
            else:
                display(widgets.HTML(value="изображение не найдено"))

        except Exception as exc:
            display(widgets.HTML(value="ошибка чтения изображения: " + str(exc)))

    with map_out:
        clear_output(wait=True)

        display(
            build_scene_map(
                row,
                show_other_frames=show_other_frames_checkbox.value,
                show_other_scenes=show_other_scenes_checkbox.value,
                show_track_points=show_track_points_checkbox.value,
            )
        )

    with table_out:
        clear_output(wait=True)
        display(widgets.HTML(value="<b>сцены выбранного трека</b>"))
        display(compact_scene_table(row["ID"]))


def on_id_change(change):
    if change["name"] == "value":
        update_scene_dropdown()
        display_selected_scene()


def on_scene_change(change):
    if change["name"] == "value":
        display_selected_scene()


def move_in_track(delta):
    indices = indices_for_id(id_dropdown.value)

    if len(indices) == 0 or scene_dropdown.value is None:
        return

    current = int(scene_dropdown.value)

    if current not in indices:
        scene_dropdown.value = indices[0]
        return

    pos = indices.index(current)
    new_pos = max(0, min(len(indices) - 1, pos + delta))

    scene_dropdown.value = int(indices[new_pos])


def move_global(delta):
    indices = global_indices()

    if len(indices) == 0 or scene_dropdown.value is None:
        return

    current = int(scene_dropdown.value)

    if current not in indices:
        new_index = indices[0]
    else:
        pos = indices.index(current)
        new_pos = max(0, min(len(indices) - 1, pos + delta))
        new_index = indices[new_pos]

    new_id = str(scenes.loc[new_index, "ID"])

    if id_dropdown.value != new_id:
        id_dropdown.value = new_id
        update_scene_dropdown(preferred_index=new_index)
    else:
        scene_dropdown.value = int(new_index)


id_dropdown.observe(on_id_change, names="value")
scene_dropdown.observe(on_scene_change, names="value")

prev_button.on_click(lambda b: move_in_track(-1))
next_button.on_click(lambda b: move_in_track(1))
global_prev_button.on_click(lambda b: move_global(-1))
global_next_button.on_click(lambda b: move_global(1))

show_other_frames_checkbox.observe(lambda change: display_selected_scene(), names="value")
show_other_scenes_checkbox.observe(lambda change: display_selected_scene(), names="value")
show_track_points_checkbox.observe(lambda change: display_selected_scene(), names="value")

scene_table_accordion = widgets.Accordion(children=[table_out])
scene_table_accordion.set_title(0, "таблица сцен выбранного трека")
scene_table_accordion.selected_index = None

scene_tab = widgets.VBox(
    [
        widgets.HTML(
            value="""
            <div style="font-family:Arial; font-size:13px; padding:8px 0;">
              <b>проверка сцен на треке</b><br>
              Сцена отображается в физическом размере: 1 пиксель изображения равен 1 км.
              Синяя точка показывает положение на полном треке в момент времени съемки.
            </div>
            """
        ),
        widgets.HBox([id_dropdown], layout=widgets.Layout(width="1500px", overflow_x="auto")),
        widgets.HBox([scene_dropdown], layout=widgets.Layout(width="1500px", overflow_x="auto")),
        widgets.HBox(
            [
                prev_button,
                next_button,
                global_prev_button,
                global_next_button,
                show_other_frames_checkbox,
                show_other_scenes_checkbox,
                show_track_points_checkbox,
            ],
            layout=widgets.Layout(width="1500px", overflow_x="auto"),
        ),
        status_out,
        widgets.HBox(
            [
                widgets.VBox(
                    [info_out, image_out],
                    layout=widgets.Layout(width="520px", overflow_x="auto"),
                ),
                widgets.VBox(
                    [map_out],
                    layout=widgets.Layout(width="1000px", overflow_x="auto"),
                ),
            ],
            layout=widgets.Layout(width="1540px", overflow_x="auto", align_items="flex-start"),
        ),
        scene_table_accordion,
    ],
    layout=widgets.Layout(width="1540px", overflow_x="auto"),
)

# 12. виджеты вкладки всех треков

all_tracks_options = [("все треки", "__all__")]

for _, row in track_summary.iterrows():
    all_tracks_options.append(
        (
            f"ID {row['ID']} | сцен={int(row['n_scenes'])} | точек={int(row['n_track_points'])} | slp={row['slp_min']:.1f}–{row['slp_max']:.1f}",
            str(row["ID"]),
        )
    )

all_tracks_dropdown = widgets.Dropdown(
    options=all_tracks_options,
    value="__all__",
    description="трек",
    layout=widgets.Layout(width="900px"),
)

all_tracks_show_points_checkbox = widgets.Checkbox(
    value=True,
    description="показывать точки треков",
    layout=widgets.Layout(width="230px"),
)

all_tracks_summary_out = widgets.Output()
all_tracks_selected_out = widgets.Output()
all_tracks_map_out = widgets.Output()
all_tracks_table_out = widgets.Output()


def update_all_tracks_tab(change=None):
    active_id = all_tracks_dropdown.value

    with all_tracks_summary_out:
        clear_output(wait=True)
        display(widgets.HTML(value="<b>обзор треков ПЦ, представленных в приложении</b>"))
        display(all_tracks_table())

    with all_tracks_selected_out:
        clear_output(wait=True)

        if active_id == "__all__":
            display(widgets.HTML(value="выбран режим просмотра всех треков."))
        else:
            selected_summary = track_summary[track_summary["ID"].astype(str) == str(active_id)].copy()
            selected_points = all_tracks_points[all_tracks_points["ID"].astype(str) == str(active_id)].copy()

            display(widgets.HTML(value="<b>сводка по выбранному треку</b>"))
            display(selected_summary)

            display(widgets.HTML(value="<b>точки выбранного трека</b>"))
            display(
                selected_points[
                    ["ID", "step", "track_datetime", "lat", "lon", "slp", "rel_vort_850_smth"]
                ].reset_index(drop=True)
            )

    with all_tracks_map_out:
        clear_output(wait=True)
        display(
            build_all_tracks_map(
                active_id=active_id,
                show_points=all_tracks_show_points_checkbox.value,
            )
        )

    with all_tracks_table_out:
        clear_output(wait=True)
        display(widgets.HTML(value="<b>сводная таблица по всем трекам</b>"))
        display(all_tracks_table())


all_tracks_dropdown.observe(update_all_tracks_tab, names="value")
all_tracks_show_points_checkbox.observe(update_all_tracks_tab, names="value")

all_tracks_accordion = widgets.Accordion(children=[all_tracks_table_out])
all_tracks_accordion.set_title(0, "сводная таблица по всем трекам")
all_tracks_accordion.selected_index = None

all_tracks_tab = widgets.VBox(
    [
        widgets.HTML(
            value="""
            <div style="font-family:Arial; font-size:13px; padding:8px 0;">
              <b>все треки со снимками</b><br>
              Здесь показаны полные треки ПЦ и точки наблюдений. Снимки в этой вкладке не подгружаются.
            </div>
            """
        ),
        widgets.HBox(
            [all_tracks_dropdown, all_tracks_show_points_checkbox],
            layout=widgets.Layout(width="1500px", overflow_x="auto"),
        ),
        widgets.HBox(
            [
                widgets.VBox(
                    [all_tracks_summary_out, all_tracks_selected_out],
                    layout=widgets.Layout(width="540px", overflow_x="auto"),
                ),
                widgets.VBox(
                    [all_tracks_map_out],
                    layout=widgets.Layout(width="980px", overflow_x="auto"),
                ),
            ],
            layout=widgets.Layout(width="1540px", overflow_x="auto", align_items="flex-start"),
        ),
        all_tracks_accordion,
    ],
    layout=widgets.Layout(width="1540px", overflow_x="auto"),
)

# 13. финальная сборка приложения с вкладками

viewer_css = """
<style>
.pl-viewer-root {
    font-family: Arial, sans-serif;
    width: 1560px;
    max-width: 100%;
    overflow-x: auto;
}
.pl-viewer-header {
    border: 1px solid #d9d9d9;
    border-radius: 12px;
    padding: 16px 18px;
    margin-bottom: 12px;
    background: #f7f7f7;
}
.pl-viewer-title {
    font-size: 22px;
    font-weight: 700;
    margin-bottom: 6px;
    color: #222;
}
.pl-viewer-subtitle {
    font-size: 14px;
    line-height: 1.45;
    color: #333;
    max-width: 1180px;
}
.pl-viewer-note {
    border: 1px solid #e0e0e0;
    border-radius: 10px;
    padding: 10px 12px;
    margin: 10px 0;
    background: #ffffff;
    font-size: 13px;
    line-height: 1.45;
    color: #333;
}
.pl-viewer-footer {
    border-top: 1px solid #ddd;
    margin-top: 12px;
    padding-top: 8px;
    font-size: 12px;
    line-height: 1.45;
    color: #444;
}
</style>
"""

viewer_header = widgets.HTML(
    value="""
    <div class="pl-viewer-root">
      <div class="pl-viewer-header">
        <div class="pl-viewer-title">
          PL_viewer: проверка пространственно-временного сопоставления сцен Sentinel-3 SLSTR с треками полярных циклонов
        </div>
        <div class="pl-viewer-subtitle">
          Приложение использует rematched consistent dataset. Положение сцены задается фактическим временем и координатами съемки.
          Давление и завихренность берутся из ассоциированной точки полного трека.
        </div>
      </div>
    </div>
    """
)

viewer_legend = widgets.HTML(
    value="""
    <div class="pl-viewer-root">
      <div class="pl-viewer-note">
        <b>Условные обозначения</b><br>
        <span style="display:inline-block;width:22px;height:10px;background:#d62728;border-radius:5px;"></span>
        выбранная сцена SLSTR и ее центр<br>
        <span style="display:inline-block;width:22px;height:10px;background:#1f77b4;border-radius:5px;"></span>
        точка положения сцены на полном треке по времени съемки<br>
        <span style="display:inline-block;width:22px;height:10px;background:#111;border-radius:5px;"></span>
        линия полного трека полярного циклона<br>
        <span style="display:inline-block;width:22px;height:10px;background:#9467bd;border-radius:5px;"></span>
        другие сцены выбранного трека, когда включена соответствующая опция
      </div>
    </div>
    """
)

source_note = """
<div style="font-family:Arial; font-size:12px; line-height:1.45; color:#333; padding:8px 0;">
  <b>Источник данных.</b>
  Сцены Sentinel-3 SLSTR и производный набор фрагментов сцен собраны автором.
  Исходные данные Sentinel-3 относятся к программе Copernicus.
  Характеристики ПЦ, включая давление и сглаженную относительную завихренность на уровне 850 гПа,
  подготовлены по данным Stoll, 2021 и Stoll, 2022.
</div>
"""

viewer_footer = widgets.HTML(
    value=f"""
    <div class="pl-viewer-root">
      <div class="pl-viewer-footer">
        {source_note}
      </div>
    </div>
    """
)

tab_switch = widgets.ToggleButtons(
    options=[
        ("проверка сцен на треке", "scene"),
        ("все треки со снимками", "tracks"),
    ],
    value="scene",
    layout=widgets.Layout(width="760px"),
)

viewer_body = widgets.Output()


def render_viewer_body(change=None):
    with viewer_body:
        clear_output(wait=True)

        if tab_switch.value == "scene":
            display(scene_tab)
            update_scene_dropdown(
                preferred_index=scene_dropdown.value
                if scene_dropdown.value is not None
                else None
            )
            display_selected_scene()
        else:
            display(all_tracks_tab)
            update_all_tracks_tab()


tab_switch.observe(render_viewer_body, names="value")

PL_viewer = widgets.VBox(
    [
        widgets.HTML(value=viewer_css),
        viewer_header,
        viewer_legend,
        tab_switch,
        viewer_body,
        viewer_footer,
    ],
    layout=widgets.Layout(width="1560px", overflow_x="auto"),
)

clear_output(wait=True)
display(PL_viewer)

update_scene_dropdown(preferred_index=None)
display_selected_scene()
update_all_tracks_tab()
render_viewer_body()

In [2]:
# сводная статистика отклонений сцен от интерполированного положения трека

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

distance_col = "distance_scene_center_to_track_time_km"
offset_col = "offset_px"
share_col = "share_width"
status_col = "position_status"
method_col = "track_time_method"

required_cols = [distance_col, status_col]
missing_cols = [col for col in required_cols if col not in scenes.columns]

if missing_cols:
    raise KeyError(
        "сначала нужно выполнить блок расчета отклонений от трека. "
        "не найдены колонки: " + ", ".join(missing_cols)
    )

overview = scenes.copy()
overview[distance_col] = pd.to_numeric(overview[distance_col], errors="coerce")

if offset_col in overview.columns:
    overview[offset_col] = pd.to_numeric(overview[offset_col], errors="coerce")

if share_col in overview.columns:
    overview[share_col] = pd.to_numeric(overview[share_col], errors="coerce")

valid_dist = overview[distance_col].dropna()

n_total = len(overview)
n_valid = int(valid_dist.shape[0])
n_missing = int(n_total - n_valid)

inside_mask = overview[status_col].astype(str).str.contains("попадает", case=False, na=False)
outside_mask = overview[status_col].astype(str).str.contains("вне сцены", case=False, na=False)

summary = pd.DataFrame(
    {
        "показатель": [
            "всего сцен",
            "сцен с рассчитанным отклонением",
            "сцен без рассчитанного отклонения",
            "среднее отклонение, км",
            "медианное отклонение, км",
            "75-й процентиль, км",
            "90-й процентиль, км",
            "95-й процентиль, км",
            "максимальное отклонение, км",
            "сцен, где трек попадает в пределы снимка",
            "доля таких сцен, %",
            "сцен, где трек вне снимка",
            "доля таких сцен, %",
        ],
        "значение": [
            n_total,
            n_valid,
            n_missing,
            round(valid_dist.mean(), 2) if n_valid else np.nan,
            round(valid_dist.median(), 2) if n_valid else np.nan,
            round(valid_dist.quantile(0.75), 2) if n_valid else np.nan,
            round(valid_dist.quantile(0.90), 2) if n_valid else np.nan,
            round(valid_dist.quantile(0.95), 2) if n_valid else np.nan,
            round(valid_dist.max(), 2) if n_valid else np.nan,
            int(inside_mask.sum()),
            round(inside_mask.sum() / n_total * 100, 2) if n_total else np.nan,
            int(outside_mask.sum()),
            round(outside_mask.sum() / n_total * 100, 2) if n_total else np.nan,
        ],
    }
)

display(Markdown("### сводная статистика отклонений снимков от треков"))
display(summary)

display(Markdown("### распределение статусов положения трека относительно снимка"))
status_summary = (
    overview[status_col]
    .fillna("нет данных")
    .value_counts()
    .rename_axis("статус")
    .reset_index(name="число сцен")
)

status_summary["доля, %"] = (status_summary["число сцен"] / n_total * 100).round(2)
display(status_summary)

if method_col in overview.columns:
    display(Markdown("### методы сопоставления сцены с треком"))
    method_summary = (
        overview[method_col]
        .fillna("нет данных")
        .value_counts()
        .rename_axis("метод")
        .reset_index(name="число сцен")
    )

    method_summary["доля, %"] = (method_summary["число сцен"] / n_total * 100).round(2)
    display(method_summary)

display(Markdown("### 15 сцен с наибольшим отклонением от трека"))

top_cols = [
    "ID",
    "image_id",
    "source_png_filename",
    "plot_datetime",
    "plot_lat",
    "plot_lon",
    "track_time_lat",
    "track_time_lon",
    distance_col,
    offset_col,
    share_col,
    status_col,
    method_col,
    "track_time_prev_step",
    "track_time_next_step",
    "track_time_fraction",
]

top_cols = [col for col in top_cols if col in overview.columns]

top_deviations = (
    overview[top_cols]
    .sort_values(distance_col, ascending=False)
    .head(15)
    .reset_index(drop=True)
)

display(top_deviations)

if "ID" in overview.columns:
    display(Markdown("### треки с наибольшими отклонениями"))

    by_track = (
        overview
        .groupby("ID", dropna=False)
        .agg(
            scenes_n=(distance_col, "size"),
            valid_n=(distance_col, lambda x: x.notna().sum()),
            mean_distance_km=(distance_col, "mean"),
            median_distance_km=(distance_col, "median"),
            max_distance_km=(distance_col, "max"),
            outside_n=(status_col, lambda x: x.astype(str).str.contains("вне сцены", case=False, na=False).sum()),
        )
        .reset_index()
    )

    by_track["mean_distance_km"] = by_track["mean_distance_km"].round(2)
    by_track["median_distance_km"] = by_track["median_distance_km"].round(2)
    by_track["max_distance_km"] = by_track["max_distance_km"].round(2)

    by_track = by_track.sort_values(
        ["max_distance_km", "mean_distance_km"],
        ascending=False,
    ).head(15)

    display(by_track)

### сводная статистика отклонений снимков от треков

,показатель,значение
0,всего сцен,393.00
1,сцен с рассчитанным отклонением,393.00
2,сцен без рассчитанного отклонения,0.00
3,"среднее отклонение, км",232.23
4,"медианное отклонение, км",211.74
5,"75-й процентиль, км",329.93
6,"90-й процентиль, км",453.16
7,"95-й процентиль, км",479.34
8,"максимальное отклонение, км",499.55
9,"сцен, где трек попадает в пределы снимка",262.00


### распределение статусов положения трека относительно снимка

,статус,число сцен,"доля, %"
0,точка трека попадает в пределы сцены,262,66.67
1,точка трека вне сцены,131,33.33


### методы сопоставления сцены с треком

,метод,число сцен,"доля, %"
0,линейная интерполяция по времени,376,95.67
1,последняя точка трека,16,4.07
2,первая точка трека,1,0.25


### 15 сцен с наибольшим отклонением от трека

,ID,image_id,source_png_filename,plot_datetime,plot_lat,plot_lon,track_time_lat,track_time_lon,distance_scene_center_to_track_time_km,offset_px,share_width,position_status,track_time_method,track_time_prev_step,track_time_next_step,track_time_fraction
0,220200112060,Solovieva_PL_crop_00159_ID_220200112060_step_9,crop_00159_ID_220200112060_step_9_4c847a8a9a.png,2020-01-30 08:58:35.517093,60.122039,175.875569,57.738266,168.488266,499.549059,499.549059,97.568176,точка трека вне сцены,линейная интерполяция по времени,8.0,9.0,0.976533
1,220201008800,Solovieva_PL_crop_00953_ID_220201008800_step_33,crop_00953_ID_220201008800_step_33_f488be8089.png,2020-10-31 08:31:13.515094,70.794705,-167.147029,74.369895,-176.239790,498.525341,498.525341,97.368231,точка трека вне сцены,линейная интерполяция по времени,32.0,33.0,0.520421
2,120200215530,Solovieva_PL_crop_00389_ID_120200215530_step_16,crop_00389_ID_120200215530_step_16_b2f4f44b36.png,2020-02-26 15:47:59.639537,77.481536,46.208508,73.449975,38.349925,498.075112,498.075112,97.280295,точка трека вне сцены,линейная интерполяция по времени,15.0,16.0,0.799900
3,120200110110,Solovieva_PL_crop_00024_ID_120200110110_step_14,crop_00024_ID_120200110110_step_14_2cc95e4780.png,2020-01-16 13:39:06.861263,63.642478,-38.574182,66.500000,-30.412976,496.597666,496.597666,96.991732,точка трека вне сцены,линейная интерполяция по времени,13.0,14.0,0.651906
4,120200317000,Solovieva_PL_crop_00739_ID_120200317000_step_7,crop_00739_ID_120200317000_step_7_a34d8023ac.png,2020-03-30 07:36:30.671233,68.813188,53.749280,68.347870,41.575559,496.217067,496.217067,96.917396,точка трека вне сцены,линейная интерполяция по времени,7.0,8.0,0.608520
5,220200210300,Solovieva_PL_crop_00429_ID_220200210300_step_11,crop_00429_ID_220200210300_step_11_c8f97fcc52.png,2020-02-28 10:44:59.634525,47.671030,156.102090,51.312525,160.062525,495.603725,495.603725,96.797603,точка трека вне сцены,линейная интерполяция по времени,10.0,11.0,0.749898
6,120200215530,Solovieva_PL_crop_00390_ID_120200215530_step_17,crop_00390_ID_120200215530_step_17_a67c86eb63.png,2020-02-26 16:48:54.367213,77.783616,46.385882,73.703776,39.111327,494.610718,494.610718,96.603656,точка трека вне сцены,линейная интерполяция по времени,16.0,17.0,0.815102
7,120200216280,Solovieva_PL_crop_00404_ID_120200216280_step_12,crop_00404_ID_120200216280_step_12_7c66c1a8b0.png,2020-02-27 12:05:28.820419,74.316361,10.841278,70.454330,18.158661,494.229447,494.229447,96.529189,точка трека вне сцены,линейная интерполяция по времени,12.0,13.0,0.091339
8,220201206210,Solovieva_PL_crop_01160_ID_220201206210_step_10,crop_01160_ID_220201206210_step_10_dc330e670a.png,2020-12-16 10:14:07.005112,50.004314,162.900429,53.250000,167.750000,492.032610,492.032610,96.100119,точка трека вне сцены,линейная интерполяция по времени,10.0,11.0,0.235279
9,220201213410,Solovieva_PL_crop_01280_ID_220201213410_step_22,crop_01280_ID_220201213410_step_22_7e3093b53b.png,2020-12-31 22:33:19.026905,54.134437,175.743200,58.500000,177.000000,491.553677,491.553677,96.006578,точка трека вне сцены,последняя точка трека,22.0,22.0,0.000000


### треки с наибольшими отклонениями

,ID,scenes_n,valid_n,mean_distance_km,median_distance_km,max_distance_km,outside_n
128,220200112060,4,4,199.03,147.78,499.55,1
142,220201008800,2,2,492.32,492.32,498.53,2
46,120200215530,3,3,484.71,494.61,498.08,3
4,120200110110,5,5,325.89,404.15,496.60,3
94,120200317000,2,2,469.41,469.41,496.22,2
135,220200210300,1,1,495.60,495.60,495.60,1
49,120200216280,1,1,494.23,494.23,494.23,1
153,220201206210,1,1,492.03,492.03,492.03,1
167,220201213410,3,3,437.46,427.10,491.55,3
137,220200304500,1,1,491.42,491.42,491.42,1


## Обзор сопоставления спутниковых сцен с треками ПЦ

В этом блоке оценивается качество пространственно-временного сопоставления спутниковых сцен Sentinel-3 SLSTR с опорными траекториями ПЦ. Для каждой сцены рассчитывается положение циклона на момент съемки и расстояние между центром спутникового фрагмента и соответствующей точкой трека.

Всего проанализировано 393 сцены. Для всех сцен удалось рассчитать отклонение от трека, пропусков в этой части нет. Среднее отклонение составляет 232.23 км, медианное — 211.74 км. 75-й процентиль равен 329.93 км, 90-й — 453.16 км, 95-й — 479.34 км. Максимальное отклонение достигает 499.55 км. Эти значения показывают, что часть сцен сопоставлена с треком с заметным пространственным смещением.

При этом в 262 случаях, или 66.67% сцен, рассчитанная точка трека попадает в пределы спутникового фрагмента. В 131 случае, или 33.33% сцен, точка трека оказывается вне сцены. Значит, около двух третей сопоставлений можно рассматривать как пригодные для визуальной проверки и дальнейшего анализа. Оставшаяся треть требует отдельного контроля.

Основной метод сопоставления — линейная интерполяция по времени между соседними точками трека. Он использован для 376 сцен, то есть для 95.67% всех случаев. Это основной и наиболее корректный вариант, поскольку время спутниковой съемки обычно не совпадает точно со временем записи в каталоге. В 16 случаях использована последняя точка трека, еще в одном случае — первая точка трека. Такие случаи возникают на границах трека, когда момент съемки расположен за пределами интервала между соседними точками.

Таблица сцен с наибольшими отклонениями показывает, что максимальные смещения близки к 500 км. Во всех 15 худших случаях точка трека находится вне сцены. Доля смещения от ширины снимка составляет примерно 95–98%, то есть рассчитанное положение циклона находится почти у внешней границы фрагмента или за ее пределами.

Агрегация по отдельным трекам помогает выявить проблемные события. Наибольшие средние отклонения наблюдаются у треков `220201008800`, `120200215530`, `120200317000`, `220200210300`, `120200216280`, `220201206210`, `220200304500` и некоторых других. Для части этих треков все сцены имеют положение трека вне снимка. Подобные случаи следует рассматривать как кандидаты на ручную проверку в приложении.

Важный результат состоит в том, что приложение позволяет отделить технически сопоставленные сцены от сцен, пригодных для содержательного анализа. Сам факт связи снимка с ID и временем трека еще не гарантирует, что центр ПЦ находится внутри фрагмента.

Итоговая оценка показывает, что алгоритм сопоставления работает воспроизводимо, для сцен рассчитано положение на треке, основной метод основан на линейной интерполяции по времени. Главная проблема связана с пространственным смещением части фрагментов.